<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">


# Python for Finance, 3rd Edition
## Chapter 22 · Efficient Markets and Hypothesis Testing

&copy; Dr. Yves J. Hilpisch<br>
AI-supported by GPT 5.x<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals
This notebook mirrors the Chapter 22 code flow in an interactive format. It
loads daily market data, studies autocorrelation and predictive regressions,
simulates a small Granger-causality system, and displays both chapter figures
inline.


### How to Use This Notebook
- Run the cells from top to bottom the first time.
- The setup cell switches into the project root so relative paths still
  work.
- The EOD dataset uses a local/remote fallback like the other Part V
  notebooks.


### Notebook Setup
Move to the project root first so the notebook can reuse the chapter's file
layout without manual path edits.


In [ ]:
from pathlib import Path
import subprocess
import sys

NOTEBOOK_SUBDIR = "notebooks"
COLAB_PACKAGES = {}
REPO_NAME = "py4fi3rd"
REPO_URL = "https://github.com/yhilpisch/py4fi3rd.git"


def _support_dir() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        support_dir = candidate / "notebooks"
        if (support_dir / "_book_notebook_support.py").exists():
            return support_dir
    if "google.colab" in sys.modules:
        root = Path("/content") / REPO_NAME
        if not root.exists():
            subprocess.run(
                ["git", "clone", "--depth", "1", REPO_URL, str(root)],
                check=True,
            )
        return root / "notebooks"
    raise RuntimeError("Could not locate notebook support helpers.")


SUPPORT_DIR = _support_dir()
if str(SUPPORT_DIR) not in sys.path:
    sys.path.insert(0, str(SUPPORT_DIR))

from _book_notebook_support import setup_notebook

CONTEXT = setup_notebook(
    notebook_subdir=NOTEBOOK_SUBDIR,
    colab_packages=COLAB_PACKAGES,
)

PROJECT_ROOT = CONTEXT["PROJECT_ROOT"]
NOTEBOOK_DIR = CONTEXT["NOTEBOOK_DIR"]
CODE_DIR = CONTEXT["CODE_DIR"]
CHAPTERS_DIR = CONTEXT["CHAPTERS_DIR"]
FIGURES_DIR = CONTEXT["FIGURES_DIR"]
DATA_DIR = CONTEXT["DATA_DIR"]

PROJECT_ROOT

## The Efficient Market Hypothesis as a Baseline
Treat EMH as a null hypothesis and start with a simple return series for a
liquid equity benchmark.


### Loading Daily SPY Returns
Load the EOD dataset with a local/remote fallback and compute daily `SPY`
returns.


In [ ]:
from pathlib import Path  # filesystem paths

import numpy as np  # numerical work
import pandas as pd  # time-series handling

LOCAL_EOD = Path.cwd() / "data" / "eod_data.csv"  # local EOD file
REMOTE_EOD = "https://hilpisch.com/eod_data.csv"  # remote fallback
source = LOCAL_EOD if LOCAL_EOD.exists() else REMOTE_EOD

prices = pd.read_csv(
    source,
    parse_dates=["Date"],
    index_col="Date",
)  # read prices with a DatetimeIndex

r_spy = prices["SPY"].pct_change().dropna()  # simple daily SPY returns
r_spy.tail()


### Checking a Few Autocorrelations
Inspect the sample autocorrelation at lag 1 and lag 5 as a first serial-
dependence diagnostic.


In [ ]:
acf_1 = r_spy.autocorr(lag=1)  # first-order autocorrelation
acf_5 = r_spy.autocorr(lag=5)  # fifth-order autocorrelation
round(float(acf_1), 4), round(float(acf_5), 4)


## Designing and Interpreting Tests
Move from descriptive autocorrelations to simple regression-based hypothesis
tests.


### Fitting a One-Lag Return Regression
Regress current returns on one lag of returns and inspect coefficients and
p-values.


In [ ]:
import statsmodels.api as sm  # regression and test utilities

r_lag1 = r_spy.shift(1).dropna()  # lagged return series
aligned = pd.concat(
    {"r": r_spy, "r_lag1": r_lag1},
    axis=1,
).dropna()  # align current and lagged returns

y = aligned["r"]  # dependent variable
X = sm.add_constant(aligned["r_lag1"])  # intercept plus lagged return
model = sm.OLS(y, X).fit()  # one-lag autoregression
model.params.round(4)


### Inspecting Regression p-Values
Check whether the lagged-return coefficient is statistically distinguishable
from zero.


In [ ]:
model.pvalues.round(4)


### Inline Figure: SPY Return Autocorrelation
Display the sample autocorrelation function of daily `SPY` returns up to 30
lags.


In [ ]:
import matplotlib as mpl  # plotting style control
import matplotlib.pyplot as plt  # plotting interface

mpl.style.use("seaborn-v0_8")  # chapter baseline style
mpl.rcParams.update({"font.family": "serif", "figure.dpi": 300})

max_lag = 30  # longest lag shown in the chart
lags = np.arange(1, max_lag + 1)  # lag labels
acf_vals = np.array(
    [r_spy.autocorr(lag=int(k)) for k in lags],
    dtype=float,
)  # sample autocorrelation values

fig, ax = plt.subplots(figsize=(6.5, 4.0))  # figure canvas
ax.bar(lags, acf_vals, width=0.8, color="C0", alpha=0.8)  # bar chart
ax.axhline(0.0, color="black", linewidth=0.8)  # zero reference line
ax.set_xlabel("Lag (trading days)")  # x-axis label
ax.set_ylabel("Sample autocorrelation")  # y-axis label
ax.grid(True, axis="y", linestyle="--", alpha=0.3)  # light grid
fig.tight_layout()  # avoid clipping
plt.show()


## Simple Regression-Based Tests for Signals
Test whether a simple momentum signal helps predict next-day returns.


### Building a 20-Day Momentum Signal
Construct the signal, align it with next-day returns, and fit the predictive
regression.


In [ ]:
window = 20  # momentum lookback window
mom_20d = r_spy.rolling(window).mean()  # rolling mean return signal
r_next = r_spy.shift(-1)  # next-day return target
aligned_sig = pd.concat(
    {"r_next": r_next, "mom_20d": mom_20d},
    axis=1,
).dropna()  # align signal and target

y = aligned_sig["r_next"]  # dependent variable
X = sm.add_constant(aligned_sig["mom_20d"])  # signal plus intercept
sig_model = sm.OLS(y, X).fit()  # predictive regression
sig_model.params.round(4)


### Inspecting Signal p-Values
Check whether the momentum coefficient is statistically strong enough to stand
out against the EMH baseline.


In [ ]:
sig_model.pvalues.round(4)


## Granger Causality in a Synthetic System
Simulate a small two-series process where one variable Granger-causes the
other by construction.


### Simulating the Driver and Response Series
Generate two short return series in which lagged `x` feeds into current `y`.


In [ ]:
import numpy as np  # array-based simulation

rng = np.random.default_rng(seed=21)  # reproducible random generator
steps = 500  # number of simulated observations
eps_x = rng.normal(0.0, 0.02, size=steps)  # shocks for x
eps_y = rng.normal(0.0, 0.02, size=steps)  # shocks for y
x = np.empty(steps)  # storage for x
y = np.empty(steps)  # storage for y
x[0] = eps_x[0]  # initial x value
y[0] = eps_y[0]  # initial y value
for t in range(1, steps):
    x[t] = 0.2 * x[t - 1] + eps_x[t]  # autoregressive x process
    y[t] = 0.5 * x[t - 1] + eps_y[t]  # y driven by lagged x
x[:5].round(4), y[:5].round(4)


### Comparing Restricted and Full Regressions
Measure how much explanatory power is gained when lagged `x` is added to the
regression for current `y`.


In [ ]:
y_lag = y[:-1]  # lagged y predictor
y_t = y[1:]  # current y target
X_y = np.column_stack([np.ones_like(y_lag), y_lag])  # restricted design
beta_y, *_ = np.linalg.lstsq(X_y, y_t, rcond=None)  # restricted fit
y_hat_y = X_y @ beta_y  # restricted fitted values
ss_tot = np.sum((y_t - y_t.mean()) ** 2)  # total sum of squares
ss_res_y = np.sum((y_t - y_hat_y) ** 2)  # restricted residual sum
r2_y = 1.0 - ss_res_y / ss_tot  # restricted R^2

x_lag = x[:-1]  # lagged x predictor
X_xy = np.column_stack(
    [np.ones_like(y_lag), y_lag, x_lag],
)  # full design matrix
beta_xy, *_ = np.linalg.lstsq(X_xy, y_t, rcond=None)  # full fit
y_hat_xy = X_xy @ beta_xy  # full fitted values
ss_res_xy = np.sum((y_t - y_hat_xy) ** 2)  # full residual sum
r2_xy = 1.0 - ss_res_xy / ss_tot  # full R^2
float(r2_y), float(r2_xy)


## Granger Causality with a Real Signal
Use `statsmodels` to test whether lagged 20-day momentum values help predict
current `SPY` returns.


### Running the Granger-Causality Tests
Align returns and momentum, then extract p-values across several lags.


In [ ]:
from statsmodels.tsa.stattools import grangercausalitytests  # Granger helper
import contextlib  # suppress helper output
import io  # in-memory text buffer

data_gc = pd.concat(
    {"r": r_spy, "mom_20d": mom_20d},
    axis=1,
).dropna()  # aligned return-signal table
max_lag = 3  # longest lag tested
buf = io.StringIO()  # suppress verbose statsmodels output
with contextlib.redirect_stdout(buf):
    tests = grangercausalitytests(
        data_gc[["r", "mom_20d"]],
        maxlag=max_lag,
    )  # run the lag-by-lag tests
pvalues = pd.Series(
    {lag: res[0]["ssr_ftest"][1] for lag, res in tests.items()},
    name="pvalue",
).sort_index()  # extract p-values by lag
pvalues.round(4)


### Inline Figure: Granger-Causality p-Values
Display the Granger-causality p-values together with the 5% reference level.


In [ ]:
lags = np.arange(1, max_lag + 1)  # lag labels
pvals_arr = np.array([tests[lag][0]["ssr_ftest"][1] for lag in lags])

fig, ax = plt.subplots(figsize=(6.5, 4.0))  # figure canvas
ax.plot(
    lags,
    pvals_arr,
    marker="o",
    color="C1",
    label="p-value",
)  # p-value line
ax.axhline(
    0.05,
    color="grey",
    linestyle="--",
    linewidth=1.0,
    alpha=0.7,
    label="5% level",
)  # significance threshold
ax.set_xlabel("Lag in Granger-causality test")  # x-axis label
ax.set_ylabel("p-value")  # y-axis label
ax.set_xticks(lags)  # integer lag ticks
ax.set_ylim(-0.1, 1.0)  # visual range
ax.grid(True, linestyle="--", alpha=0.3)  # light grid
ax.legend(loc="upper right", frameon=False)  # compact legend
fig.tight_layout()  # avoid clipping
plt.show()


## Out-of-Sample and Walk-Forward Evaluation
In-sample tests screen candidate signals, but out-of-sample evaluation is what
connects statistical evidence to trading or portfolio decisions.


## From Statistical Evidence to Portfolio and Trading Decisions
Treat p-values and regressions as tools for a research loop, not as final
proof that a signal is economically actionable.


<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">
